# Vera C. Rubin Observatory Scheduler Reports

The reports here are indexed below by night (dayobs) and instrument, with links to the following reports in different columns:

- **prenight** is a report describing a simulation of the night completed the previous morning. It includes basic statistics of the visits from the simulation (total science visits taken, etc.), a dynamic map of the visits, and tools for plotting various visit parameters against time. In addition, the report includes visualizationn of the reward functions that led to each FBS selection, which can be helpful in understading what led to the FBS selecting any given visit.
- **multiprenight** is a report that compares the nominal pre-night simulation described above and a few variants of it (e.g. simulating a late start). I includes figures like tables showing what fraction of visits that occur on one simulation also occur in other variants.  (Those interested in more details for the variants can use the Times Square interface to generate prenight reports like the one described above for any of the simulations completed; see the index [here](https://usdf-rsp.slac.stanford.edu/times-square/github/lsst/schedview_notebooks/prenight/prenight_sims): fill in the night you are interested in on the left, then select the desired sim from the resultant table.)
- **nightsum** is a scheduler-focused summary of the night, including basic paramaters of the night (twilight times, etc.), basic statistics for the visits of the night (e.g. number of science visits, mean gap times between visits, etc.), a tool for plotting assorted visit parameters against time, an interactive visit map, a handful of maps for different metrics such as photometric depth, a DDF cadence plot showing when DDFs have been observed over the past year and how deep each night's observations went, and a text table of exposures.
- **compareprenight** is a report that compares the set of science visits actually taken during a night with the pre-night simulations of that night, including plots in which parameter values are plotted over time, with simulation visits overplotted on completed visits, statistics on the numbers of visits each simulation has in common with the completed visits and the offsets in corresponding visit's start times, and a plot of completed vs. simulated visit start times for visits they have in common.

`eff_time` as reported in the statistics in the table is a measure of the photometric depth, as described in section 7 of [DMTN-296](https://dmtn-296.lsst.io/), using reference m5 values from [SMTN-002](https://smtn-002.lsst.io/).

In [ ]:
"""
# To render this notebook to html, in the current directory with the current python environment

jupyter nbconvert \
    --to html \
    --execute \
    --no-input \
    --ExecutePreprocessor.kernel_name=python3 \
    --ExecutePreprocessor.startup_timeout=3600 \
    --ExecutePreprocessor.timeout=3600 \
    schedview_reports_toc.ipynb

# The public USDF page served at
#    https://s3df.slac.stanford.edu/data/rubin/sim-data/schedview/reports
# is found at
#    /sdf/group/rubin/web_data/sim-data/schedview/reports/index.html

# You can set up the conda environment used to automatically generate these pages with
#    conda activate /sdf/data/rubin/shared/scheduler/envs/prenight
"""
pass

In [ ]:
from IPython.display import HTML
import sys

In [ ]:
#sched_source = 'env'
sched_source = 'shared'
#sched_source = 'devel'
match sched_source:
    case 'shared':
        if os.path.exists('/sdf/data/rubin/shared/scheduler/packages'):
            sys.path.insert(0, "/sdf/data/rubin/shared/scheduler/packages/rubin_scheduler")
            sys.path.insert(0, "/sdf/data/rubin/shared/scheduler/packages/rubin_sim")
            sys.path.insert(0, "/sdf/data/rubin/shared/scheduler/packages/schedview")
    case 'devel':
        if os.path.exists('/sdf/data/rubin/user/neilsen/devel'):
            sys.path.insert(0, "/sdf/data/rubin/user/neilsen/devel/rubin_scheduler")
            sys.path.insert(0, "/sdf/data/rubin/user/neilsen/devel/rubin_sim")
            sys.path.insert(0, "/sdf/data/rubin/user/neilsen/devel/schedview")
    case _:
        # Use whatever is in the kernel python environment
        pass

In [ ]:
from pathlib import Path
from schedview.util import config_logging_for_reports
import logging
config_logging_for_reports(logging.ERROR)

In [ ]:
import pandas as pd
from rubin_scheduler.site_models import Almanac
from schedview.reports import find_reports, make_report_link_table, make_report_rss_feed
from schedview.collect.visits import cached_read_visits
from schedview.compute import smallsum
from schedview import DayObs

In [ ]:
schedview_cache_dir = "/sdf/group/rubin/web_data/sim-data/schedview/cache"
schedview_report_base = "/sdf/group/rubin/web_data/sim-data/schedview/reports"
schedview_url_base = "https://s3df.slac.stanford.edu/data/rubin/sim-data/schedview/reports"
dayobs_today = DayObs.from_date('today')

In [ ]:
visits = cached_read_visits(dayobs_today, 'lsstcam', schedview_cache_dir)

In [ ]:
reports = find_reports(schedview_report_base, url_base=schedview_url_base)
report_columns = ("prenight", "multiprenight", "nightsum", "compareprenight")
report_link_table =  make_report_link_table(reports, report_columns=report_columns, visits=visits)
display(HTML(report_link_table))

In [ ]:
instruments = reports.index.get_level_values(level='instrument').unique()
for instrument in instruments:
    inst_reports = reports.loc[instrument, :]
    report_names = inst_reports['report'].unique()
    for report_name in report_names:
        latest_dayobs = inst_reports.reset_index().set_index('report').loc[report_name, 'dayobs'].max()
        latest_url = reports.set_index('report', append=True).loc[(instrument, latest_dayobs, report_name), 'url']
        latest_file = schedview_report_base + latest_url[len(schedview_url_base):]
        latest_file_link = Path(schedview_report_base).joinpath(report_name, instrument, 'latest.html')
        try:
            try:
                latest_file_link.hardlink_to(latest_file)
            except FileExistsError:
                latest_file_link.unlink()
                latest_file_link.hardlink_to(latest_file)
        except (PermissionError, OSError):
            print("Could not update link to latest versions")

In [ ]:
# Writes the rss feed to a publicly readable USDF site.
for report in 'nightsum', 'prenight':
    for instrument in 'lsstcam', 'latiss':
        rss_feed_fname = f'/sdf/group/rubin/web_data/sim-data/schedview/{instrument}_{report}.rss'
        these_reports = reports.reset_index().query(f'instrument=="{instrument}" and report=="{report}"').set_index(['instrument', 'dayobs'])
        rss_tree = make_report_rss_feed(these_reports, fname=rss_feed_fname, visits=visits, max_days=7, title=f"{instrument} {report}")

In [ ]:
# Writes the rss feed to a publicly readable USDF site.
rss_feed_fname = '/sdf/group/rubin/web_data/sim-data/schedview/public_schedview_reports.rss'
rss_tree = make_report_rss_feed(reports, fname=rss_feed_fname, visits=visits)